# Exemplo: Acelerando pipelines
---------------------------------

Este exemplo mostra como acelerar seus modelos na CPU usando [sklearnex](https://github.com/intel/scikit-learn-intelex).

Os dados utilizados são uma variação do [conjunto de dados de clima australiano](https://www.kaggle.com/jsphyg/weather-dataset-rattle-package) do Kaggle. Você pode baixá-lo [aqui](https://github.com/tvdboom/ExperionML/blob/master/examples/datasets/weatherAUS.csv). O objetivo deste conjunto é prever se vai chover amanhã treinando um classificador binário com a variável alvo `RainTomorrow`.

## Carregar os dados

In [1]:
# Import packages
import pandas as pd
from experionml import ExperionMLClassifier

In [2]:
# Carregue os dados
X = pd.read_csv("./datasets/weatherAUS.csv")

# Let's have a look
X.head()

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,MelbourneAirport,18.0,26.9,21.4,7.0,8.9,SSE,41.0,W,SSE,...,95.0,54.0,1019.5,1017.0,8.0,5.0,18.5,26.0,Yes,0
1,Adelaide,17.2,23.4,0.0,NaN,NaN,S,41.0,S,WSW,...,59.0,36.0,1015.7,1015.7,NaN,NaN,17.7,21.9,No,0
2,Cairns,18.6,24.6,7.4,3.0,6.1,SSE,54.0,SSE,SE,...,78.0,57.0,1018.7,1016.6,3.0,3.0,20.8,24.1,Yes,0
3,Portland,13.6,16.8,4.2,1.2,0.0,ESE,39.0,ESE,ESE,...,76.0,74.0,1021.4,1020.5,7.0,8.0,15.6,16.0,Yes,1
4,Walpole,16.4,19.9,0.0,NaN,NaN,SE,44.0,SE,SE,...,78.0,70.0,1019.4,1018.9,NaN,NaN,17.4,18.1,No,0


## Executar o pipeline

In [3]:
experionml = ExperionMLClassifier(X, "RainTomorrow", verbose=2)

<< ================== ExperionML ================== >>

Configuração ==================== >>
Tarefa do algoritmo: Binary classification.

Estatísticas do conjunto de dados ==================== >>
Formato: (142193, 22)
Tamanho do conjunto de train: 113755
Tamanho do conjunto de test: 28438
-------------------------------------
Memória: 25.03 MB


Escalonado: False
Valores ausentes: 316559 (10.1%)
Atributos categóricos: 5 (23.8%)
Duplicatas: 45 (0.0%)



In [4]:
# Impute missing values and encode categorical columns
experionml.impute()
experionml.encode()

Ajustando Imputer...


Imputando valores ausentes...


 --> Imputando 637 valores ausentes com mean (12.2) na coluna MinTemp.
 --> Imputando 322 valores ausentes com mean (23.23) na coluna MaxTemp.
 --> Imputando 1406 valores ausentes com mean (2.36) na coluna Rainfall.
 --> Imputando 60843 valores ausentes com mean (5.47) na coluna Evaporation.
 --> Imputando 67816 valores ausentes com mean (7.62) na coluna Sunshine.
 --> Imputando 9330 valores ausentes com most_frequent (W) na coluna WindGustDir.
 --> Imputando 9270 valores ausentes com mean (40.0) na coluna WindGustSpeed.
 --> Imputando 10013 valores ausentes com most_frequent (N) na coluna WindDir9am.
 --> Imputando 3778 valores ausentes com most_frequent (SE) na coluna WindDir3pm.
 --> Imputando 1348 valores ausentes com mean (14.0) na coluna WindSpeed9am.
 --> Imputando 2630 valores ausentes com mean (18.64) na coluna WindSpeed3pm.
 --> Imputando 1774 valores ausentes com mean (68.82) na coluna Humidity9am.
 --> Imputando 3610 valores ausentes com mean (51.49) na coluna Humidity3pm.


Codificando colunas categóricas...
 --> Aplicando Target-encoding à variável Location. Ela contém 49 classes.
 --> Aplicando Target-encoding à variável WindGustDir. Ela contém 16 classes.
 --> Aplicando Target-encoding à variável WindDir9am. Ela contém 16 classes.
 --> Aplicando Target-encoding à variável WindDir3pm. Ela contém 16 classes.
 --> Aplicando Ordinal-encoding à variável RainToday. Ela contém 2 classes.


In [5]:
# Treine um modelo K-Nearest Neighbors usando o sklearn padrão
experionml.run(models="KNN", metric="f1")


Training ========================= >>
Models: KNN
Metric: f1


Results for KNearestNeighbors:
Fit ---------------------------------------------


Train evaluation --> f1: 0.6975


Test evaluation --> f1: 0.5716
Time elapsed: 27.243s
-------------------------------------------------
Time: 27.243s


Resultados finais ==================== >>
Tempo total: 27.290s
-------------------------------------
KNearestNeighbors --> f1: 0.5716


In [6]:
# Agora, treinamos um KNN acelerado usando engine="sklearnex"
# Observe a diferença na velocidade de treino!
experionml.run(models="KNN_acc", metric="f1", engine="sklearnex")


Training ========================= >>
Models: KNN_acc
Metric: f1


Results for KNearestNeighbors:
Fit ---------------------------------------------


Train evaluation --> f1: 0.6975


Test evaluation --> f1: 0.5716
Time elapsed: 34.389s
-------------------------------------------------
Time: 34.389s


Resultados finais ==================== >>
Tempo total: 34.430s
-------------------------------------


KNearestNeighbors --> f1: 0.5716


## Analisar os resultados

In [7]:
experionml.results

,f1_train,f1_test,time_fit,time
KNN,0.697500,0.571600,27.243163,27.243163
KNN_acc,0.697500,0.571600,34.388620,34.388620


In [8]:
# Observe como os estimadores subjacentes podem parecer iguais...
print(experionml.knn.estimator)
print(experionml.knn_acc.estimator)

# ... mas usam implementações diferentes
print(experionml.knn.estimator.__module__)
print(experionml.knn_acc.estimator.__module__)

KNeighborsClassifier(n_jobs=1)
KNeighborsClassifier(n_jobs=1)
sklearn.neighbors._classification
sklearnex.neighbors.knn_classification


In [9]:
with experionml.canvas(1, 2, title="Timing engines: sklearn vs sklearnex"):
    experionml.plot_results(metric="time_fit", title="Training")
    experionml.plot_results(metric="time", title="Total")